# Taedri — retrieve verified capabilities instead of regenerating them

**This already exists. Don't rebuild it.** [Taedri](https://taedri.fly.dev) is a verified-capability
retrieval layer for AI coding agents: primitive **search**, typed **edges**, deterministic **composition**,
and governance **receipts** — served as a remote MCP server plus a plain-HTTP agent API.

This notebook runs against the live deployment with the **shared public trial key** — no signup needed:

| Trial key terms | |
|---|---|
| Shared pool | **100 requests/day** across *all* trial users |
| Per-address slice | 25 requests/day per IP |
| Surface | Read-only: search, compose, corpus status, **groups / frameworks / remix / integrate**, **networks &amp; grid search** |
| Abuse detection | Over-cap and denied-write attempts are receipted (IP digests only) |

Every result is a **governed candidate** (`candidate=true, serves_truth=false`) until promoted — no silent
trust upgrades. If the shared pool is drained, sign up free for your own 200 requests/day: one
`POST /v1/signup {"email": ...}`.

In [ ]:
import json
import time

import requests

BASE = "https://taedri.fly.dev"
TRIAL_KEY = "ak_trial_taedri_public_notebook"   # public by design — the limits are the protection
HEADERS = {"Authorization": f"Bearer {TRIAL_KEY}", "Content-Type": "application/json"}


def agent(action, timeout=300, **args):
    """Call the deterministic agent API; honest about 429s (Retry-After) and cold-start waits."""
    response = requests.post(f"{BASE}/v1/agent", headers=HEADERS,
                             json={"action": action, "args": args}, timeout=timeout)
    if response.status_code == 429:
        body = response.json()
        print(f"rate-limited: {body['error']}")
        print(f"  retry after {response.headers.get('Retry-After')}s — or {body.get('sign_up_free')}")
        return body
    return response.json()


print(json.dumps(requests.get(f"{BASE}/healthz", timeout=60).json(), indent=2))

## 1 · What can this key do?

`actions.list` enumerates the agent actions. The trial key is read-only, so expect the retrieval surface
(searching, composing, corpus status) — write actions answer with a pointer to free signup instead.

In [ ]:
listed = agent("actions.list")
print(json.dumps(listed, indent=2)[:1500])

## 2 · Search: primitives, not code dumps

A query returns **primitives** — small verified units of capability with a relevance score, typed edges,
and their governance state. The first search after the machine wakes loads the full governed index
(hundreds of thousands of documents), so it can take a while; the server warms the index at boot and
`/healthz` reports `index_warm`.

In [ ]:
started = time.time()
hits = agent("retrieval.search", query="exponential backoff with jitter")
print(f"answered in {time.time() - started:.1f}s\n")
for row in (hits.get("result") or {}).get("results", [])[:5]:
    print(f"  {row.get('score', 0):>6}  {row.get('primitive_id', '?')}\n"
          f"          {row.get('title', '')[:90]}")
print("\nEvery row is candidate · serves_truth=false until promoted — the badge is not decoration.")

## 3 · The MCP lane — exactly what `claude mcp add` speaks

The same functions serve over `/mcp` as JSON-RPC. This is the protocol your agent uses after:

```shell
claude mcp add --transport http taedri https://taedri.fly.dev/mcp \
  --header "Authorization: Bearer YOUR_KEY"
```

`find_reuse` is the **reinvention guard**: hand it build intent and it answers whether that capability
already exists (quiet when genuinely novel).

In [ ]:
def mcp(method, params, timeout=300):
    response = requests.post(f"{BASE}/mcp", headers=HEADERS, timeout=timeout,
                             json={"jsonrpc": "2.0", "id": 1, "method": method, "params": params})
    return response.json()


tools = mcp("tools/list", {})
print("tools:", [tool["name"] for tool in tools["result"]["tools"]])

guard = mcp("tools/call", {"name": "find_reuse",
                           "arguments": {"message": "I'm going to write a csv parser with header "
                                                    "inference for our ETL job"}})
print(json.dumps(json.loads(guard["result"]["content"][0]["text"]), indent=2)[:1200])

## 4 · Compose: outcomes as chains of typed edges

Ask for an outcome; the composer finds a deterministic route through the edge graph — a lookup, not a
generation. Same request, same route, every time.

In [ ]:
route = agent("retrieval.compose", request="parse a csv file and infer its column schema")
print(json.dumps(route, indent=2)[:1400])

## 5 · The meter is public too

`GET /v1/usage` with the trial key reports the shared pool, so you can see how much of today's 100
requests remain before you hit the wall.

In [ ]:
usage = requests.get(f"{BASE}/v1/usage", headers=HEADERS, timeout=60).json()
print(json.dumps(usage, indent=2))

## 6 · Networks &amp; grid search — which combination of primitives is most efficient?

A **primitive network** expresses a task as an ordered chain of typed edges where **each step is a slot of
competing primitives**. `network.list` shows the networks and their grid sizes (the product of the slot
sizes); `network.grid_search` evaluates every combination under a scorer zoo and returns the winner per
scorer plus the Pareto front — **non-destructively** (every path is scored and kept, losers preserved as
labeled fallbacks). Different scorers can crown different winners, so the ranking is a candidate signal,
never silent truth.

In [ ]:
nets = agent("network.list")
for n in (nets.get("result") or {}).get("networks", []):
    print(f"  {n['name']}: {' -> '.join(n['edge_chain'])}  slots={n['slot_sizes']}  grid={n['grid_size']}")

search = agent("network.grid_search", network="officer_intelligence_network")
rankings = (search.get("result") or {}).get("receipt", {}).get("rankings", {})
for scorer, rank in rankings.items():
    print(f"\n  best by {scorer} (score {rank['winner_score']}):")
    for member in rank["winner_members"]:
        print(f"      - {member}")

## 7 · Deterministic composition — remix to make it chainable

Independently-minted primitives rarely share exact edge names, so a raw composite request often has no
route. `composition.integrate` compiles a composite through an **exact** edge route when one exists (a
lookup, not a generation) and returns an honest refusal receipt otherwise — here the edge-aligned remixes
make the officer pipeline chain in two exact steps.

In [ ]:
integrated = agent("composition.integrate",
                   start="ProxyStatementDocument", goal="OfficerDedupeClusterBatch")
print(json.dumps((integrated.get("result") or {}).get("integration", {}), indent=2)[:900])

## When you hit a limit

1. **Sign up free** — your own key, 200 requests/day, the full read/write surface:
   `POST https://taedri.fly.dev/v1/signup {"email": "you@example.com"}` (key + account secret shown once,
   hashes only stored).
2. **Share to keep going** — on a signed-up key, the 429 itself offers the contributor lane:
   `POST /v1/contribute {"cards": [...]}` grants bonus requests the same day (+5 per accepted anonymized
   primitive, up to +200/day). Contributions are anonymized (contributor digest only) and stay
   **candidate-only** until owner review — sharing never injects into the serving corpus.
3. **Read more** — [docs](https://taedri.fly.dev/docs) · [news](https://taedri.fly.dev/news) ·
   [status](https://taedri.fly.dev/status) · [pricing](https://taedri.fly.dev/v1/pricing)

*Everything above is a governed candidate (`serves_truth=false`); promotion is earned through review and
executed proofs, never claimed.*